# Full Benchmark Evaluation

**Goal:** Unified comparison of all oversampling methods under identical conditions.

| Method | Type |
|---|---|
| No Resampling | Baseline |
| Class Weights | Baseline |
| SMOTE | Classical |
| CTGAN | Generative |
| TVAE | Generative |

**Classifiers:** Logistic Regression, Random Forest  
**Metrics:** F1, PR-AUC, MCC

> CTGAN/TVAE synthetic data is cached to CSV after first run to avoid retraining.

In [ ]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import matplotlib.cm as cm
import seaborn as sns
from pathlib import Path

from sklearn.model_selection import train_test_split
from sklearn.preprocessing import StandardScaler, OrdinalEncoder
from sklearn.linear_model import LogisticRegression
from sklearn.ensemble import RandomForestClassifier
from sklearn.metrics import (
    f1_score, average_precision_score, matthews_corrcoef,
    classification_report, confusion_matrix,
    precision_recall_curve, PrecisionRecallDisplay
)
from imblearn.over_sampling import SMOTENC
from sdv.single_table import CTGANSynthesizer, TVAESynthesizer
from sdv.metadata import SingleTableMetadata

plt.style.use('seaborn-v0_8-whitegrid')
sns.set_palette('husl')

DATA_PATH         = Path('../data/healthcare-dataset-stroke-data.csv')
CTGAN_CACHE       = Path('../data/synthetic_ctgan.csv')
TVAE_CACHE        = Path('../data/synthetic_tvae.csv')
RANDOM_STATE      = 42

NUMERICAL   = ['age', 'avg_glucose_level', 'bmi']
CATEGORICAL = ['hypertension', 'heart_disease', 'gender', 'ever_married',
               'work_type', 'Residence_type', 'smoking_status']

## 1. Load & Preprocess

In [ ]:
df = pd.read_csv(DATA_PATH)
df = df.drop(columns=['id'])
df = df[df['gender'] != 'Other'].copy()
df['bmi'] = df['bmi'].fillna(df['bmi'].median())

# Raw split (for generative models)
train_raw, test_raw = train_test_split(
    df, test_size=0.2, random_state=RANDOM_STATE, stratify=df['stroke']
)

# Encoded split (for classifiers and SMOTE)
enc = OrdinalEncoder()
df_enc = df.copy()
df_enc[CATEGORICAL] = enc.fit_transform(df[CATEGORICAL])

X = df_enc[NUMERICAL + CATEGORICAL].values
y = df_enc['stroke'].values
cat_indices = list(range(len(NUMERICAL), len(NUMERICAL) + len(CATEGORICAL)))

X_train, X_test, y_train, y_test = train_test_split(
    X, y, test_size=0.2, random_state=RANDOM_STATE, stratify=y
)

scaler = StandardScaler()
n = len(NUMERICAL)
X_train_s = np.hstack([scaler.fit_transform(X_train[:, :n]), X_train[:, n:]])
X_test_s  = np.hstack([scaler.transform(X_test[:, :n]),      X_test[:, n:]])

n_majority    = (y_train == 0).sum()
n_minority    = (y_train == 1).sum()
n_to_generate = n_majority - n_minority

print(f'Train: {X_train_s.shape} | Stroke: {y_train.sum()} ({y_train.mean()*100:.1f}%)')
print(f'Test:  {X_test_s.shape}  | Stroke: {y_test.sum()} ({y_test.mean()*100:.1f}%)')
print(f'Samples to generate: {n_to_generate}')

## 2. Generate Synthetic Data

SMOTE is applied inline. CTGAN/TVAE synthetic data is cached to CSV — training runs only on the first execution.

In [ ]:
# --- SMOTE ---
smotenc = SMOTENC(categorical_features=cat_indices, random_state=RANDOM_STATE)
X_train_smote, y_train_smote = smotenc.fit_resample(X_train_s, y_train)
print(f'SMOTE: {X_train_smote.shape} | Stroke: {y_train_smote.sum()} ({y_train_smote.mean()*100:.1f}%)')

In [ ]:
def build_augmented(synthetic_raw):
    """Combine raw train data with synthetic samples, encode and scale."""
    combined = pd.concat([train_raw, synthetic_raw], ignore_index=True)
    enc2 = OrdinalEncoder(handle_unknown='use_encoded_value', unknown_value=-1)
    combined[CATEGORICAL] = enc2.fit_transform(combined[CATEGORICAL])
    X_tr = combined[NUMERICAL + CATEGORICAL].values
    y_tr = combined['stroke'].values

    test_enc = test_raw.copy()
    test_enc[CATEGORICAL] = enc2.transform(test_raw[CATEGORICAL])
    X_te = test_enc[NUMERICAL + CATEGORICAL].values

    sc2 = StandardScaler()
    X_tr_s = np.hstack([sc2.fit_transform(X_tr[:, :n]), X_tr[:, n:]])
    X_te_s = np.hstack([sc2.transform(X_te[:, :n]),     X_te[:, n:]])
    return X_tr_s, y_tr, X_te_s

In [ ]:
# --- CTGAN ---
if CTGAN_CACHE.exists():
    synthetic_ctgan = pd.read_csv(CTGAN_CACHE)
    print(f'CTGAN: loaded from cache ({len(synthetic_ctgan)} samples)')
else:
    minority_train = train_raw[train_raw['stroke'] == 1].copy()
    metadata = SingleTableMetadata()
    metadata.detect_from_dataframe(minority_train)
    for col in NUMERICAL:
        metadata.update_column(col, sdtype='numerical')
    for col in CATEGORICAL + ['stroke']:
        metadata.update_column(col, sdtype='categorical')

    ctgan = CTGANSynthesizer(metadata, epochs=300, verbose=True)
    ctgan.fit(minority_train)
    synthetic_ctgan = ctgan.sample(num_rows=n_to_generate)
    synthetic_ctgan['stroke'] = 1
    synthetic_ctgan.to_csv(CTGAN_CACHE, index=False)
    print(f'CTGAN: trained and cached ({len(synthetic_ctgan)} samples)')

X_train_ctgan, y_train_ctgan, X_test_ctgan = build_augmented(synthetic_ctgan)

In [ ]:
# --- TVAE ---
if TVAE_CACHE.exists():
    synthetic_tvae = pd.read_csv(TVAE_CACHE)
    print(f'TVAE: loaded from cache ({len(synthetic_tvae)} samples)')
else:
    minority_train = train_raw[train_raw['stroke'] == 1].copy()
    metadata = SingleTableMetadata()
    metadata.detect_from_dataframe(minority_train)
    for col in NUMERICAL:
        metadata.update_column(col, sdtype='numerical')
    for col in CATEGORICAL + ['stroke']:
        metadata.update_column(col, sdtype='categorical')

    tvae = TVAESynthesizer(metadata, epochs=300, verbose=True)
    tvae.fit(minority_train)
    synthetic_tvae = tvae.sample(num_rows=n_to_generate)
    synthetic_tvae['stroke'] = 1
    synthetic_tvae.to_csv(TVAE_CACHE, index=False)
    print(f'TVAE: trained and cached ({len(synthetic_tvae)} samples)')

X_train_tvae, y_train_tvae, X_test_tvae = build_augmented(synthetic_tvae)

## 3. Evaluate All Methods

In [ ]:
results = []
trained_models = {}  # store for later deep-dive

def evaluate(method, clf_name, clf, X_tr, y_tr, X_te, y_te):
    clf.fit(X_tr, y_tr)
    y_pred = clf.predict(X_te)
    y_prob = clf.predict_proba(X_te)[:, 1]
    f1     = f1_score(y_te, y_pred)
    pr_auc = average_precision_score(y_te, y_prob)
    mcc    = matthews_corrcoef(y_te, y_pred)
    results.append({'Method': method, 'Classifier': clf_name,
                    'F1': round(f1,4), 'PR-AUC': round(pr_auc,4), 'MCC': round(mcc,4)})
    trained_models[f'{method}_{clf_name}'] = (clf, y_prob)
    print(f'[{method}] {clf_name} | F1: {f1:.4f} | PR-AUC: {pr_auc:.4f} | MCC: {mcc:.4f}')
    return clf

LR = lambda: LogisticRegression(max_iter=1000, random_state=RANDOM_STATE)
RF = lambda: RandomForestClassifier(n_estimators=200, random_state=RANDOM_STATE)
LR_w = lambda: LogisticRegression(max_iter=1000, class_weight='balanced', random_state=RANDOM_STATE)
RF_w = lambda: RandomForestClassifier(n_estimators=200, class_weight='balanced', random_state=RANDOM_STATE)

y_test_eval = y_test  # same for No Resampling, Class Weights, SMOTE

print('=== No Resampling ===')
evaluate('No Resampling', 'Logistic Regression', LR(),   X_train_s,     y_train,       X_test_s,    y_test_eval)
evaluate('No Resampling', 'Random Forest',        RF(),   X_train_s,     y_train,       X_test_s,    y_test_eval)

print('\n=== Class Weights ===')
evaluate('Class Weights', 'Logistic Regression', LR_w(), X_train_s,     y_train,       X_test_s,    y_test_eval)
evaluate('Class Weights', 'Random Forest',        RF_w(), X_train_s,     y_train,       X_test_s,    y_test_eval)

print('\n=== SMOTE ===')
evaluate('SMOTE',         'Logistic Regression', LR(),   X_train_smote, y_train_smote, X_test_s,    y_test_eval)
evaluate('SMOTE',         'Random Forest',        RF(),   X_train_smote, y_train_smote, X_test_s,    y_test_eval)

print('\n=== CTGAN ===')
y_test_ctgan = test_raw['stroke'].values
evaluate('CTGAN',         'Logistic Regression', LR(),   X_train_ctgan, y_train_ctgan, X_test_ctgan, y_test_ctgan)
evaluate('CTGAN',         'Random Forest',        RF(),   X_train_ctgan, y_train_ctgan, X_test_ctgan, y_test_ctgan)

print('\n=== TVAE ===')
y_test_tvae = test_raw['stroke'].values
evaluate('TVAE',          'Logistic Regression', LR(),   X_train_tvae,  y_train_tvae,  X_test_tvae,  y_test_tvae)
evaluate('TVAE',          'Random Forest',        RF(),   X_train_tvae,  y_train_tvae,  X_test_tvae,  y_test_tvae)

## 4. Results Table

In [ ]:
results_df = pd.DataFrame(results)
results_df = results_df.sort_values('F1', ascending=False).reset_index(drop=True)
print(results_df.to_string(index=False))

## 5. Visualisations

### 5.1 Bar Charts — F1 / PR-AUC / MCC

In [ ]:
methods   = ['No Resampling', 'Class Weights', 'SMOTE', 'CTGAN', 'TVAE']
metrics   = ['F1', 'PR-AUC', 'MCC']
clfs      = ['Logistic Regression', 'Random Forest']
colors    = {'Logistic Regression': '#2196F3', 'Random Forest': '#F44336'}
x         = np.arange(len(methods))
width     = 0.35

fig, axes = plt.subplots(1, 3, figsize=(16, 5))

for ax, metric in zip(axes, metrics):
    for i, clf_name in enumerate(clfs):
        vals = [
            results_df[(results_df['Method'] == m) & (results_df['Classifier'] == clf_name)][metric].values[0]
            for m in methods
        ]
        offset = (i - 0.5) * width
        bars = ax.bar(x + offset, vals, width, label=clf_name,
                      color=colors[clf_name], alpha=0.85)
        for bar, v in zip(bars, vals):
            ax.text(bar.get_x() + bar.get_width()/2, bar.get_height() + 0.005,
                    f'{v:.3f}', ha='center', va='bottom', fontsize=7)
    ax.set_title(metric, fontsize=13, fontweight='bold')
    ax.set_xticks(x)
    ax.set_xticklabels(methods, rotation=20, ha='right', fontsize=9)
    ax.set_ylim(0, ax.get_ylim()[1] * 1.1)
    ax.legend(fontsize=8)

plt.suptitle('All Methods — Benchmark Comparison', fontsize=15, fontweight='bold')
plt.tight_layout()
plt.savefig('../data/benchmark_bar.png', dpi=150, bbox_inches='tight')
plt.show()

### 5.2 Heatmap

In [ ]:
fig, axes = plt.subplots(1, 2, figsize=(14, 5))

for ax, clf_name in zip(axes, clfs):
    subset = results_df[results_df['Classifier'] == clf_name].set_index('Method')[metrics]
    subset = subset.reindex(methods)
    sns.heatmap(subset, annot=True, fmt='.3f', cmap='YlOrRd',
                vmin=0, vmax=1, ax=ax, linewidths=0.5)
    ax.set_title(clf_name, fontsize=12, fontweight='bold')
    ax.set_ylabel('')

plt.suptitle('Heatmap — Methods × Metrics', fontsize=14, fontweight='bold')
plt.tight_layout()
plt.savefig('../data/benchmark_heatmap.png', dpi=150, bbox_inches='tight')
plt.show()

### 5.3 Precision-Recall Curves

PR curves show the precision-recall trade-off across all thresholds — more informative than a single F1 score on imbalanced data.

In [ ]:
fig, axes = plt.subplots(1, 2, figsize=(14, 5))
method_colors = {
    'No Resampling': '#9E9E9E',
    'Class Weights': '#FF9800',
    'SMOTE':         '#2196F3',
    'CTGAN':         '#F44336',
    'TVAE':          '#4CAF50',
}

# Map each method to its correct y_test
y_test_map = {
    'No Resampling': y_test,
    'Class Weights': y_test,
    'SMOTE':         y_test,
    'CTGAN':         y_test_ctgan,
    'TVAE':          y_test_tvae,
}

for ax, clf_name in zip(axes, clfs):
    for method in methods:
        _, y_prob = trained_models[f'{method}_{clf_name}']
        y_true    = y_test_map[method]
        prec, rec, _ = precision_recall_curve(y_true, y_prob)
        pr_auc = average_precision_score(y_true, y_prob)
        ax.plot(rec, prec, label=f'{method} (AUC={pr_auc:.3f})',
                color=method_colors[method], linewidth=1.8)
    ax.set_xlabel('Recall')
    ax.set_ylabel('Precision')
    ax.set_title(clf_name, fontsize=12, fontweight='bold')
    ax.legend(fontsize=8)

plt.suptitle('Precision-Recall Curves', fontsize=14, fontweight='bold')
plt.tight_layout()
plt.savefig('../data/pr_curves.png', dpi=150, bbox_inches='tight')
plt.show()

## 6. Best Model Deep Dive

Identify the best model by F1 and analyse in detail.

In [ ]:
best = results_df.iloc[0]
print(f'Best model: [{best["Method"]}] {best["Classifier"]}')
print(f'  F1: {best["F1"]} | PR-AUC: {best["PR-AUC"]} | MCC: {best["MCC"]}')

best_clf, best_prob = trained_models[f'{best["Method"]}_{best["Classifier"]}']
best_y_test = y_test_map[best['Method']]
best_pred   = best_clf.predict(
    X_test_ctgan if best['Method'] == 'CTGAN' else
    X_test_tvae  if best['Method'] == 'TVAE'  else X_test_s
)

print('\n' + classification_report(best_y_test, best_pred, target_names=['No Stroke', 'Stroke']))

### 6.1 Confusion Matrix

In [ ]:
cm_vals = confusion_matrix(best_y_test, best_pred)

fig, ax = plt.subplots(figsize=(5, 4))
sns.heatmap(cm_vals, annot=True, fmt='d', cmap='Blues',
            xticklabels=['No Stroke', 'Stroke'],
            yticklabels=['No Stroke', 'Stroke'], ax=ax)
ax.set_xlabel('Predicted')
ax.set_ylabel('Actual')
ax.set_title(f'Confusion Matrix\n[{best["Method"]}] {best["Classifier"]}',
             fontsize=12, fontweight='bold')
plt.tight_layout()
plt.savefig('../data/confusion_matrix.png', dpi=150, bbox_inches='tight')
plt.show()

### 6.2 Feature Importance (Random Forest only)

In [ ]:
# Find best Random Forest model
rf_results = results_df[results_df['Classifier'] == 'Random Forest']
best_rf_method = rf_results.iloc[0]['Method']
best_rf, _ = trained_models[f'{best_rf_method}_Random Forest']

feature_names = NUMERICAL + CATEGORICAL
importances   = best_rf.feature_importances_
sorted_idx    = np.argsort(importances)[::-1]

fig, ax = plt.subplots(figsize=(8, 5))
ax.bar(range(len(feature_names)), importances[sorted_idx], color='#2196F3', alpha=0.85)
ax.set_xticks(range(len(feature_names)))
ax.set_xticklabels([feature_names[i] for i in sorted_idx], rotation=35, ha='right')
ax.set_title(f'Feature Importance — Random Forest [{best_rf_method}]',
             fontsize=12, fontweight='bold')
ax.set_ylabel('Importance')
plt.tight_layout()
plt.savefig('../data/feature_importance.png', dpi=150, bbox_inches='tight')
plt.show()

## 7. Summary

Key findings from the benchmark:

In [ ]:
print('=== Benchmark Summary ===')
print(f"\nBest overall (F1): [{results_df.iloc[0]['Method']}] {results_df.iloc[0]['Classifier']}")
print(f"  F1={results_df.iloc[0]['F1']} | PR-AUC={results_df.iloc[0]['PR-AUC']} | MCC={results_df.iloc[0]['MCC']}")

print('\nRanking by F1:')
for _, row in results_df.iterrows():
    print(f"  {row['Method']:<18} {row['Classifier']:<22} F1={row['F1']}")

# Does generative AI beat SMOTE?
smote_f1 = results_df[results_df['Method'] == 'SMOTE']['F1'].max()
gen_f1   = results_df[results_df['Method'].isin(['CTGAN', 'TVAE'])]['F1'].max()
print(f'\nSMOTE best F1:      {smote_f1:.4f}')
print(f'Generative best F1: {gen_f1:.4f}')
print(f'Generative AI beats SMOTE: {gen_f1 > smote_f1}')